In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os
import h5py
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Read Dataset

In [ ]:
class CustomNormalize:
    def __call__(self, tensor):
        min_val = tensor.min()
        max_val = tensor.max()
        epsilon = 1e-8
        return (tensor - min_val) / (max_val - min_val + epsilon)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    CustomNormalize()
])


In [ ]:
train_dataset = datasets.ImageFolder(os.path.join("D:/Workspace_1/medical-image-ai/dataset/pretrain_data", 'train'), transform=transform)
val_dataset = datasets.ImageFolder(os.path.join("D:/Workspace_1/medical-image-ai/dataset/pretrain_data", 'val'), transform=transform)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
model = models.resnet18()
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
num_classes = len(train_dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# 训练循环
for epoch in range(25):
    # 训练阶段
    model.train()
    train_loss = 0.0
    train_correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()

    # 验证阶段
    model.eval()
    val_correct = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()

    train_acc = 100. * train_correct / len(train_dataset)
    val_acc = 100. * val_correct / len(val_dataset)

    print(f'Epoch [{epoch+1}/{25}], '
          f'Train Loss: {train_loss/len(train_loader):.4f}, '
          f'Train Acc: {train_acc:.2f}%, '
              f'Val Acc: {val_acc:.2f}%')

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'classes': train_dataset.classes,
    'class_to_idx': train_dataset.class_to_idx
}, 'MRI_pretrained.pth')

print(f"saved")